# Model Improvement

The baseline and tuned models achieved approximately 80% validation accuracy, which is below the project target of 85%.

The previous experiments used only information available before departure. Hyperparameter tuning did not produce a meaningful improvement, indicating that the current features contain limited predictive information.

This notebook investigates an alternative prediction scenario in which arrival delay is predicted immediately after the flight departs. Therefore, actual departure information such as `DEP_DELAY` and `DEP_TIME` can be included.

This experiment is interpreted as an after-departure arrival-delay prediction model rather than a pre-departure prediction model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

In [ ]:
from pathlib import Path


def find_project_root():
    """
    Locate the project root regardless of whether the notebook
    is launched from the project root or the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Keep the data and notebooks folders inside the same project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)
df.head()

## Feature Selection

This experiment predicts whether a flight will arrive more than 15 minutes late immediately after departure.

Therefore, actual departure information can be included:

- `DEP_TIME`: actual departure time
- `DEP_DELAY`: departure delay in minutes

Arrival-related variables and delay-cause variables remain excluded because they would reveal information about the target outcome.

In [ ]:
required_columns = [
    "DEP_TIME",
    "DEP_DELAY",
    "IS_DELAYED",
    "YEAR"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

In [ ]:
selected_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST",

    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",

    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "QUARTER",

    "DEP_HOUR",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

target = "IS_DELAYED"

print("Number of selected features:", len(selected_features))

In [ ]:
model_df = df[selected_features + [target]].copy()

print("Modeling dataset shape:", model_df.shape)
model_df.head()

In [ ]:
missing_summary = (
    model_df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

## Chronological Data Splitting

The same chronological split used in the baseline modeling notebook is maintained:

- Training set: 2019–2021
- Validation set: 2022
- Test set: 2023

This allows the after-departure model to be compared fairly with the previous pre-departure models.

In [ ]:
train_df = model_df[
    model_df["YEAR"] <= 2021
].copy()

validation_df = model_df[
    model_df["YEAR"] == 2022
].copy()

test_df = model_df[
    model_df["YEAR"] == 2023
].copy()

In [ ]:
X_train = train_df[selected_features]
y_train = train_df[target]

X_validation = validation_df[selected_features]
y_validation = validation_df[target]

X_test = test_df[selected_features]
y_test = test_df[target]

In [ ]:
split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Years": [
        "2019–2021",
        "2022",
        "2023"
    ],
    "Rows": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "Delay Rate": [
        y_train.mean(),
        y_validation.mean(),
        y_test.mean()
    ]
})

split_summary

## Preprocessing Pipeline

The preprocessing pipeline applies different transformations according to feature type:

- Numerical features are standardized using `StandardScaler`.
- Low-cardinality categorical features are encoded using `OneHotEncoder`.
- High-cardinality categorical features are encoded using `TargetEncoder`.

The preprocessing steps and classifier are combined into one pipeline to ensure that all transformations are learned only from the training data.

In [ ]:
numerical_features = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "YEAR",
    "MONTH",
    "DAY",
    "QUARTER",
    "DEP_HOUR",
    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

one_hot_features = [
    "DAY_OF_WEEK",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY"
]

target_encode_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST"
]

print("Numerical features:", len(numerical_features))
print("One-hot features:", len(one_hot_features))
print("Target-encoded features:", len(target_encode_features))

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

one_hot_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

target_transformer = Pipeline(
    steps=[
        (
            "target_encoder",
            TargetEncoder(
                target_type="binary",
                random_state=42
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "onehot",
            one_hot_transformer,
            one_hot_features
        ),
        (
            "target",
            target_transformer,
            target_encode_features
        )
    ],
    remainder="drop"
)

preprocessor

## Model Evaluation

The model is evaluated using Accuracy, Precision, Recall, F1-score, and ROC-AUC.

Because the target variable is imbalanced, Accuracy alone is insufficient. Recall and F1-score are particularly important for determining whether the model successfully identifies delayed flights.

In [ ]:
def evaluate_model(model, X, y, model_name="Model"):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(
        y,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y,
        y_pred,
        zero_division=0
    )
    roc_auc = roc_auc_score(y, y_prob)

    results = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ],
        "Score": [
            accuracy,
            precision,
            recall,
            f1,
            roc_auc
        ]
    })

    print("\n" + "=" * 60)
    print(f"{model_name} Performance")
    print("=" * 60)

    display(results)

    print("\nClassification Report\n")
    print(
        classification_report(
            y,
            y_pred,
            zero_division=0
        )
    )

    ConfusionMatrixDisplay.from_predictions(
        y,
        y_pred
    )

    plt.title(
        f"{model_name} Confusion Matrix"
    )

    plt.show()

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    }

## After-Departure XGBoost Model

An XGBoost classifier is trained using the original scheduled flight features together with actual departure information.

The most important additional feature is `DEP_DELAY`, which records how many minutes late or early the flight departed. Since this value is only available after departure, this model predicts arrival delay immediately after the flight has departed.

In [ ]:
after_departure_xgb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

after_departure_xgb_pipeline

In [ ]:
print(
    "Training after-departure XGBoost model..."
)

after_departure_xgb_pipeline.fit(
    X_train,
    y_train
)

print("Training completed.")

In [ ]:
after_departure_xgb_results = evaluate_model(
    model=after_departure_xgb_pipeline,
    X=X_validation,
    y=y_validation,
    model_name="After-Departure XGBoost"
)

In [ ]:
after_departure_xgb_results

## After-Departure Model Results

The after-departure XGBoost model achieved a validation accuracy of approximately 93.15%, exceeding the project requirement of 85%.

The model also achieved:

- Precision: 92.28%
- Recall: 72.37%
- F1-score: 81.12%
- ROC-AUC: 93.69%

These results represent a substantial improvement over the pre-departure models. The improvement is mainly associated with the inclusion of actual departure information, particularly `DEP_DELAY`.

Departure delay is strongly related to arrival delay because flights that depart late often have limited opportunity to recover the lost time during the journey.

This experiment changes the prediction scenario. The model predicts arrival delay immediately after departure rather than before departure. Therefore, the results must not be interpreted as pre-departure predictions.

In [ ]:
model_scenario_comparison = pd.DataFrame([
    {
        "Model": "Pre-Departure XGBoost",
        "Prediction Moment": "Before departure",
        "Accuracy": 0.797004,
        "Precision": 0.510611,
        "Recall": 0.029782,
        "F1": 0.056281,
        "ROC_AUC": 0.635091
    },
    {
        "Model": after_departure_xgb_results["Model"],
        "Prediction Moment": "Immediately after departure",
        "Accuracy": after_departure_xgb_results["Accuracy"],
        "Precision": after_departure_xgb_results["Precision"],
        "Recall": after_departure_xgb_results["Recall"],
        "F1": after_departure_xgb_results["F1"],
        "ROC_AUC": after_departure_xgb_results["ROC_AUC"]
    }
])

model_scenario_comparison

## Model Improvement Conclusion

The pre-departure models achieved approximately 80% validation accuracy and showed limited ability to identify delayed flights. Hyperparameter tuning alone did not produce a substantial improvement.

The prediction scenario was therefore changed from pre-departure prediction to after-departure prediction. Actual departure information, including `DEP_DELAY` and `DEP_TIME`, was added to the model.

The after-departure XGBoost model achieved approximately 93.15% validation accuracy, 81.12% F1-score, and 93.69% ROC-AUC. It therefore exceeded the required accuracy threshold while maintaining strong Precision and meaningful Recall for delayed flights.

The after-departure XGBoost model is selected as the final model configuration. The untouched 2023 test set will be used in the next notebook for final evaluation.